In [ ]:
import time
import sys
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from xgboost import XGBClassifier as XGBoostClassifier   # 'pip install xgboost' to install

# ---- XGBoost ----

# Dataset processing
df = pd.read_csv("dataset_packet_multiclass8_n10000.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

# Train the model
xgb = XGBoostClassifier()

start_time = time.process_time()
xgb.fit(X_train, y_train, verbose=False)
training_time = time.process_time() - start_time

# Model statistics
y_pred = xgb.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
full_report = classification_report(y_test, y_pred)
num_features = xgb.n_features_in_
size_kb = sys.getsizeof(pickle.dumps(xgb)) / 1024

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

In [ ]:

# Dataset processing
df = pd.read_csv("dataset_packet_multiclass8_n1000000.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

# Train the model
xgb = XGBoostClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=8,
    tree_method='hist',
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=0,
    eval_metric='mlogloss',
    n_jobs=-1
)

start_time = time.process_time()
xgb.fit(X_train, y_train, verbose=False)
training_time = time.process_time() - start_time

# Model statistics
y_pred = xgb.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
full_report = classification_report(y_test, y_pred)
num_features = xgb.n_features_in_
size_kb = sys.getsizeof(pickle.dumps(xgb)) / 1024

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

***
### Random Grid Search with Cross Validaton

In [ ]:
import time
import sys
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from xgboost import XGBClassifier as XGBoostClassifier

from sklearn.model_selection import RandomizedSearchCV

# Load the dataset
df = pd.read_csv("dataset_packet_multiclass8_n100000.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

xgb = XGBoostClassifier(
    random_state=0,
    n_jobs=-1
)

param_distribution = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],           # Similar to 'step size'
    'max_depth': [3, 6, 8, 12, 20],                    # Control RAM/Overfitting
    'min_child_weight': [1, 5, 10, 50],                # Similar to 'min_samples_leaf'
    'colsample_bytree': [0.3, 0.5, 0.7, 0.8, 1.0],     # Equivalent to 'max_features'
    'subsample': [0.1, 0.3, 0.5, 0.8, 1.0],            # Equivalent to 'max_samples'
    'gamma': [0, 0.1, 0.5, 1, 5],                      # Minimum loss reduction for a split
    'reg_alpha': [0, 0.01, 0.1, 1],                    # L1 regularization
    'reg_lambda': [1, 2, 5, 10],                       # L2 regularization
    'booster': ['gbtree', 'dart']                      # gbtree is standard; dart is slower/accurate
}

xgb_random = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_distribution,
    n_iter=50,           # Number of random combinations to try
    cv=3,                # 3-fold cross validation
    verbose=2, 
    random_state=0,
    scoring='accuracy',
    refit=True,
    n_jobs=-1            # Use all cores for the search itself
)

start_wall_time = time.perf_counter()
xgb_random.fit(X_train, y_train)
total_wall_time = time.perf_counter() - start_wall_time

best_xgb = xgb_random.best_estimator_

y_pred = best_xgb.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
size_kb = sys.getsizeof(pickle.dumps(best_xgb)) / 1024
training_time = xgb_random.cv_results_['mean_fit_time'][xgb_random.best_index_]

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

print(f"\n[+] Best Parameters: {xgb_random.best_params_}")
print(f"[+] Best Cross-Validation Accuracy: {xgb_random.best_score_:.4f}")